In [1]:
import os
from fastapi import FastAPI
from pydantic import BaseModel
from typing import List
from langchain_chroma.vectorstores import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_ollama import OllamaLLM
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

EMBEDDING_MODEL: str = 'llama3' # If changing this, make sure to also import the proper libraries
GENERATIVE_MODEL: str = 'llama3' 
CHUNK_SIZE: int = 512
CHUNK_OVERLAP: int = 20
PERSIST_DIRECTORY = 'db'
LLM_NUM_PREDICT = 128
LLM_TEMPERATURE = 0.5

app = FastAPI(title='simple rag')

class QueryRequest(BaseModel):
    user_query: str

class QueryResponse(BaseModel):
    query: str
    response: str
    sources: str

In [2]:
def embeddings():
    '''
        Embeds the text
        Parameters:
            - Embedding Model
    '''
    embeddings = OllamaEmbeddings (
        model=EMBEDDING_MODEL,
    )
    return embeddings

In [3]:
def extract_text(path):
    ''' 
        Given a path, chunks it and returns a dataset
        Parameters: 
            - Chunk Size
            - Chunk Overlap
    '''
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        length_function=len,
        is_separator_regex=False,
    )

    dataset = []

    with os.scandir(path) as es:
        for e in es:
            if e.is_file() and e.name.endswith('.md'):
                with open(e.path, 'r', encoding='utf-8') as file:
                    file_metadata = {'filename': file.name} 
                    dataset.extend(text_splitter.create_documents(texts=[file.read()], metadatas=[file_metadata])) 

    return dataset

In [4]:
def store_text(dataset):
    '''
        Stores text from dataset into persistent directory (db in our case)
    '''
    Chroma.from_documents(
        documents=[dataset[i] for i in range(len(dataset))],
        embedding=embeddings(),
        persist_directory=PERSIST_DIRECTORY,
    )

In [ ]:
def generate_text(query):
    '''
        Given a query, uses the stored text as a retriever and generates an answer.
        Parameters:
            - Model
            - Temperature
            - Num Predict
    '''
    model = OllamaLLM(
        model = GENERATIVE_MODEL,
        temperature = LLM_TEMPERATURE, 
        num_predict = LLM_NUM_PREDICT,
    )

    retriever = Chroma(persist_directory=PERSIST_DIRECTORY, embedding_function=embeddings()).as_retriever(search_kwargs={'k': 5, 'score_threshold': 0.5})

    prompt = ChatPromptTemplate.from_template(""" Answer using ONLY the context below. If the context doesn't contain the answer,
        then mention that it doesn't. Cite the source file from the metadata as ONLY a file name ONLY in a section of its own.

        Context: {context}
    
        Question: {question}
    """)

    RAG = (
        {
        'context': retriever,
        'question': RunnablePassthrough()
        }
        | prompt
        | model
    )

    answer = RAG.invoke(query)
    return answer

In [6]:
store_text(extract_text('sample-data'))

In [7]:
print(generate_text('What is an ally?'))

According to the context, an ally is someone who is willing to take action in support of another person, in order to remove barriers that impede that person from contributing their skills and talents in the workplace or community.

Source file: being-an-ally.md


In [ ]:
# from langchain.retrievers import ContextualCompressionRetriever
# from langchain.retrievers.document_compressors import CrossEncoderReranker
# from langchain_community.cross_encoders import HuggingFaceCrossEncoder

# base = Chroma(persist_directory=PERSIST_DIRECTORY, embedding_function=embeddings()) \
#     .as_retriever(search_kwargs={'k': 20})
# reranker = CrossEncoderReranker(
#     model=HuggingFaceCrossEncoder(model_name='BAAI/bge-reranker-base'), top_n=4)
# retriever = ContextualCompressionRetriever(base_compressor=reranker, base_retriever=base)
